## Visibilidad
#### A. Atributos Públicos (Acceso Directo)
En Python, la práctica estándar y más limpia para comenzar a diseñar una clase es exponer los atributos directamente. En lugar de llenar el código de métodos de acceso (*getters* y *setters*) innecesarios antes de que se requieran, se permite la lectura y escritura directa de los atributos.
*   **Ejemplo:** `self.name = name`.

#### B. Atributos Protegidos (Convención de un guion bajo `_`)
Si deseas indicar que un atributo o método es de uso interno y que no debería modificarse o accederse desde fuera de la clase, se le antepone un **guion bajo único**. 
*   **Ejemplo:** `self._balance = initial_amount`.
*   **Regla:** Técnicamente, Python permite acceder y modificar este atributo desde el exterior (`objeto._balance`), pero hacerlo rompe la convención del lenguaje y se considera una "mala práctica" o un mal diseño.

#### C. Atributo privado (`__`)
Esto impide que los programadores alteren accidentalmente el atributo mediante llamadas directas como cuenta.__balance = 99999 y evita colisiones de nombres al utilizar herencia múltiple

### Ejemplo Práctico 1: El Enfoque de Seguridad Clásico
Imagina un sistema bancario donde el saldo de una cuenta es una información crítica. No se puede permitir que un desarrollador modifique el saldo directamente desde el exterior de la clase sin pasar por validaciones de negocio previas (como evitar retiros de más dinero del disponible o depósitos con importes negativos).

A continuación se muestra cómo encapsular este comportamiento de forma segura:

In [ ]:
class Account:
    def __init__(self, name, account_number, initial_amount):
        self.name = name
        self.account_number = account_number
        # Atributo privado para proteger el saldo contra modificaciones externas
        self.__balance = initial_amount

    # Método de acceso (Getter): Interfaz de solo lectura segura
    def get_balance(self):
        """Devuelve el balance actual sin permitir su modificación directa."""
        return self.__balance

    # Operación de negocio controlada (Método de modificación o Mutator)
    def deposit(self, amount):
        """Permite depositar dinero validando que el monto sea positivo."""
        if amount > 0:
            self.__balance += amount
        else:
            raise ValueError("El monto a depositar debe ser mayor a cero.")

    # Operación de negocio controlada
    def withdraw(self, amount):
        """Permite retirar dinero validando que existan fondos suficientes."""
        if 0 < amount <= self.__balance:
            self.__balance -= amount
        else:
            raise ValueError("Monto inválido o fondos insuficientes.")

# demo
# Creación del objeto cuenta
mi_cuenta = Account("Juan Pérez", "987654321", 5000)

# Intento de modificar el saldo directamente (Fallará o no surtirá efecto)
try:
    mi_cuenta.__balance = 1000000  # Intentamos alterar el saldo directamente
except AttributeError:
    pass

# El saldo sigue intacto gracias al Name Mangling
print(f"inicial {mi_cuenta.get_balance()}")  # Salida: 5000

# Operaciones seguras y validadas a través de su interfaz pública
mi_cuenta.deposit(1500)
print(f"después del depósito {mi_cuenta.get_balance()}")  # Salida: 6500

mi_cuenta.withdraw(2000)
print(f"después del retiro {mi_cuenta.get_balance()}")  # Salida: 4500

inicial 5000
después del depósito 6500
después del retiro 4500


### encapsulamiento gradual
@property


In [2]:
class Person:
    def __init__(self, age):
        # Al usar 'self.age = age' en el constructor, se activa automáticamente el setter
        self.age = age

    @property
    def age(self):
        """Getter: Se ejecuta de forma invisible cuando leemos 'persona.age'."""
        return self._age

    @age.setter
    def age(self, value):
        """Setter: Se ejecuta de forma invisible cuando asignamos 'persona.age = valor'."""
        if 18 <= value <= 99:
            self._age = value
        else:
            raise ValueError("La edad debe estar comprendida entre 18 y 99 años.")

#demo
# Instanciamos la clase de forma normal
usuario = Person(25)

# Accedemos a la propiedad como si fuera un simple atributo de datos
print(usuario.age)  # Salida: 25 (ejecuta el método getter de forma implícita)

# Modificamos el valor de manera limpia
usuario.age = 42    # Ejecuta el método setter de forma implícita
print(usuario.age)  # Salida: 42

# Intentamos ingresar un valor inválido que viola las reglas de encapsulamiento
try:
    usuario.age = 150  # Lanza ValueError: La edad debe estar comprendida entre 18 y 99 años.
except ValueError as e:
    print(e)

25
42
La edad debe estar comprendida entre 18 y 99 años.


## **@property**

El decorador **`@property`** es una de las herramientas más elegantes y utilizadas en la programación orientada a objetos en Python. Su función principal es **convertir un método de una clase para que pueda ser accedido como si fuera un atributo de datos simple** (es decir, sin necesidad de usar paréntesis al llamarlo).

Para definir una propiedad completa en Python, se emplean decoradores que dividen las responsabilidades en tres acciones básicas: Lectura (Getter), Escritura (Setter) y Eliminación (Deleter).

In [5]:
from datetime import date

class Persona:
    def __init__(self, nombre, nacimiento):
        self.nombre = nombre
        # Esto llamará automáticamente al método setter durante la inicialización
        self.nacimiento = nacimiento 

    # 1. El GETTER (Lectura): Define la propiedad externa 'nacimiento'
    @property
    def nacimiento(self):
        """Devuelve la fecha de nacimiento de la persona."""
        print("Accediendo al getter...")
        return self._nacimiento

    # 2. El SETTER (Escritura): Valida y escribe el dato interno
    @nacimiento.setter
    def nacimiento(self, value):
        print(f"Intentando guardar la fecha: {value}...")
        if value > date.today():
            raise ValueError("No puede nacer en el futuro")
        # Guardamos el valor real en una variable protegida con un guion bajo
        self._nacimiento = value

    # 3. El DELETER (Eliminación - Opcional)
    @nacimiento.deleter
    def nacimiento(self):
        print("Borrando el atributo nacimiento...")
        del self._nacimiento

#demo
# Creación de instancia (Ejecuta el setter implícitamente en el __init__)
p = Persona("Carlos", date(1995, 5, 17))
# Salida: Intentando guardar la fecha: 1995-05-17...

# Acceso de lectura (Llama al getter de forma invisible, ¡sin paréntesis!)
print(p.nacimiento)
# Salida:
# Accediendo al getter...
# 1995-05-17

# Modificación de datos (Llama al setter)
p.nacimiento = date(2010, 8, 20)
# Salida: Intentando guardar la fecha: 2010-08-20...

# Intento de ingresar un dato inválido (Lanza la excepción)
try:
    p.nacimiento = date(2028, 12, 15)  # Fecha en el futuro
except ValueError as e:
    print(f"Error capturado: {e}")
# Salida:
# Intentando guardar la fecha: 2028-12-15...
# Error capturado: No puede nacer en el futuro

Intentando guardar la fecha: 1995-05-17...
Accediendo al getter...
1995-05-17
Intentando guardar la fecha: 2010-08-20...
Intentando guardar la fecha: 2028-12-15...
Error capturado: No puede nacer en el futuro
